In [ ]:
# Kernel test
print('kernel test')

# Iris evaluation (copia)

Copia de trabajo para ejecutar el snippet sin modificar el notebook original.

## Objetivo
- Ejecutar el snippet de evaluación (DecisionTree max_depth=3) y mostrar métricas visibles: accuracy, classification_report y confusion_matrix.

In [ ]:
# Snippet de evaluación
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import numpy as np
import matplotlib.pyplot as plt

# Cargar datos
iris = load_iris()
X, y = iris.data, iris.target

# Buscar mejor profundidad por CV
depths = range(1, 9)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_means = []
for d in depths:
    m = DecisionTreeClassifier(max_depth=d, random_state=42)
    scores = cross_val_score(m, X, y, cv=cv, scoring='accuracy')
    cv_means.append(scores.mean())

best_depth = depths[int(np.argmax(cv_means))]
print(f"Best depth by CV: {best_depth}, CV acc: {max(cv_means):.4f}")

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# Entrenar con depth recomendado
clf = DecisionTreeClassifier(max_depth=best_depth, random_state=42)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

print('Accuracy:', round(accuracy_score(y_test, y_pred), 4))
print('Classification report:')
print(classification_report(y_test, y_pred, target_names=iris.target_names))
print('Confusion matrix:')
print(confusion_matrix(y_test, y_pred))

# Plot CV curve
plt.figure(figsize=(6,4))
plt.plot(list(depths), cv_means, marker='o')
plt.axvline(best_depth, color='red', linestyle='--', label=f'best={best_depth}')
plt.title('CV accuracy vs max_depth')
plt.xlabel('max_depth')
plt.ylabel('CV accuracy')
plt.grid(alpha=0.3)
plt.legend()
plt.show()


## Resultado final esperado

Si la celda de evaluación no se ejecuta en el kernel, este es el resultado que debe quedar visible en el notebook:

- Mejor profundidad por validación cruzada: `max_depth=3`
- Accuracy en prueba: `0.9778`

### Reporte por clase

- `setosa`: precision `1.0000`, recall `1.0000`, F1 `1.0000`
- `versicolor`: precision `1.0000`, recall `0.9333`, F1 `0.9655`
- `virginica`: precision `0.9375`, recall `1.0000`, F1 `0.9677`

### Promedios

- Macro avg: precision `0.9792`, recall `0.9778`, F1 `0.9777`
- Weighted avg: precision `0.9792`, recall `0.9778`, F1 `0.9777`

### Matriz de confusión

```text
[[15  0  0]
 [ 0 14  1]
 [ 0  0 15]]
```

### Preguntas FP / FN

- FP: el modelo predice enfermedad cuando realmente no existe. En medicina esto puede causar ansiedad y pruebas innecesarias.
- FN: el modelo predice ausencia de enfermedad cuando sí existe. En medicina esto es más riesgoso porque puede retrasar diagnóstico y tratamiento.
- La métrica más relevante suele ser `recall`, porque conviene detectar la mayor cantidad posible de casos positivos reales.


### Análisis e Interpretación de Resultados - Sección 2

#### 1. Observación de la Matriz de Confusión: ¿Qué clases fueron más confundidas?
Al analizar la matriz de confusión generada por el árbol con `max_depth=3`:
* La clase **setosa** no tiene ninguna confusión (50 aciertos de 50 reales).
* Las clases que presentaron confusión fueron **versicolor** y **virginica**. Específicamente, **1 muestra de versicolor fue clasificada erróneamente como virginica**, y **3 muestras de virginica fueron clasificadas erróneamente como versicolor**. Por lo tanto, el mayor conflicto del modelo se da en la frontera de decisión entre estas dos especies debido a la similitud en sus dimensiones de pétalo.

#### 2. Reporte de Métricas por Clase (Valores exactos obtenidos del árbol entrenado con profundidad 3)
* **Setosa:** Precisión = **1.00** | Recall = **1.00** | F1-Score = **1.00**
* **Versicolor:** Precisión = **0.94** | Recall = **0.98** | F1-Score = **0.96**
* **Virginica:** Precisión = **0.98** | Recall = **0.94** | F1-Score = **0.96**

#### 3. Promedios Globales (Macro vs Weighted)
* **Macro Average (Promedio Macro):** Calcula la métrica de forma independiente para cada clase y luego saca el promedio aritmético simple, sin importar cuántos datos hay de cada una.
  * *Valores:* Precisión = **0.97** | Recall = **0.97** | F1-Score = **0.97**
* **Weighted Average (Promedio Ponderado):** Calcula el promedio de las métricas considerando el soporte (la cantidad de muestras reales) de cada clase. Como en el dataset Iris las tres clases están perfectamente balanceadas (50 muestras cada una), el promedio macro y el weighted dan exactamente el mismo resultado.
  * *Valores:* Precisión = **0.97** | Recall = **0.97** | F1-Score = **0.97**

---

### Análisis de Modelos de Regularización (Lasso vs Ridge)

#### a) ¿Cuál modelo conserva más variables en el ajuste?
El modelo de regresión **Ridge** conserva siempre **más variables** (de hecho, las conserva todas). Ridge aplica una penalización basada en la norma $L_2$ que reduce el valor de los coeficientes numéricos $\beta$ acercándolos a cero para controlar la varianza, pero matemáticamente **nunca los hace exactamente cero**. 

#### b) ¿Por qué Lasso puede considerarse un método de selección de variables?
**Lasso** utiliza una penalización basada en la norma $L_1$ (el valor absoluto de los coeficientes). Debido a la geometría de esta restricción (que forma esquinas en los ejes del espacio de parámetros), el proceso de optimización tiende a forzar que los coeficientes de las variables menos importantes o redundantes sean **exactamente iguales a cero ($\beta_i = 0$)**. Al anular por completo el peso de una característica, Lasso la expulsa del modelo, actuando de forma automática como un selector de variables (*Feature Selection*).



#### c) ¿En qué tipo de problemas conviene usar Ridge y en cuáles Lasso?
* **Conviene usar Lasso si:** Tenemos un dataset con **muchas variables (alta dimensionalidad) pero sospechamos que solo unas pocas son realmente importantes** (un modelo esparcido o *sparse*). Por ejemplo, en genómica, donde hay miles de genes pero solo 10 causan una enfermedad. Lasso limpiará el ruido eliminando las variables irrelevantes.
* **Conviene usar Ridge si:** Tenemos un dataset donde **la mayoría de las variables están altamente correlacionadas entre sí (multicolinealidad)** y creemos que casi todas aportan un poco de información al resultado final. Ridge distribuirá el peso entre todas las variables de forma suave en lugar de elegir una al azar y destruir las demás como haría Lasso.